In [ ]:
# Install required packages
!pip install -q transformers datasets torch torchvision torchaudio \
                matplotlib seaborn scikit-learn tqdm pandas numpy

import sys
from pathlib import Path

# Add your project to path
sys.path.append(r'C:/Child Guardian GP2/child guardian code')
PROJECT_ROOT = Path().resolve().parent
sys.path.append(str(PROJECT_ROOT))

# Import from core.training
from core.training import train_epoch, evaluate, save_model, ChatDataset
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler  # Added Dataset here

import os
import pandas as pd
import numpy as np
import torch
from torch.utils.data import DataLoader, WeightedRandomSampler
from transformers import DistilBertTokenizer, DistilBertForSequenceClassification
from torch.optim import AdamW
from transformers import get_linear_schedule_with_warmup
from sklearn.metrics import f1_score
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
import json

# Setup device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Paths
DATA_PATH = os.getenv('DATA_PATH')
MODEL_PATH = os.getenv('MODEL_PATH')
os.makedirs(MODEL_PATH, exist_ok=True)

# Load data
train_df = pd.read_csv(f'{DATA_PATH}/goemotions_train.csv')
val_df   = pd.read_csv(f'{DATA_PATH}/goemotions_val.csv')
test_df  = pd.read_csv(f'{DATA_PATH}/goemotions_test.csv')


# Subsample neutral in train
neutral_col = 'neutral'
if neutral_col in train_df.columns:
    neutral_mask = train_df[neutral_col] == 1
    neutral_df = train_df[neutral_mask]
    non_neutral_df = train_df[~neutral_mask]

    # Keep 30% of neutral
    neutral_df = neutral_df.sample(frac=0.3, random_state=42)

    # Recombine
    train_df = pd.concat([non_neutral_df, neutral_df]).reset_index(drop=True)

# Load emotion list
with open(f'{DATA_PATH}/goemotions_emotions.txt', 'r') as f:
    emotion_cols = [line.strip() for line in f.readlines()]
emotion_cols = [col for col in emotion_cols if col != 'example_very_unclear']

# Define labels **after all train preprocessing**
train_labels = train_df[emotion_cols].values.astype(np.float32)
val_labels   = val_df[emotion_cols].values.astype(np.float32)
test_labels  = test_df[emotion_cols].values.astype(np.float32)
print(f"\nTrain: {train_df.shape}, Val: {val_df.shape}, Test: {test_df.shape}")
print(f"Number of emotion classes: {len(emotion_cols)}")

# Verify emotion columns
available_cols = [col for col in emotion_cols if col in train_df.columns]
if len(available_cols) < len(emotion_cols):
    missing = set(emotion_cols) - set(available_cols)
    print(f"\n⚠️ Warning: Missing emotion columns: {missing}")
    emotion_cols = available_cols

print(f"\nTrain: {train_df.shape}, Val: {val_df.shape}, Test: {test_df.shape}")
print(f"Number of emotion classes: {len(emotion_cols)}")



print(f"\nUsing {len(emotion_cols)} emotion classes")

# Initialize tokenizer
tokenizer = DistilBertTokenizer.from_pretrained('distilbert-base-uncased')

train_encodings = tokenizer(
    train_df['text'].tolist(),
    truncation=True,
    padding='max_length',
    max_length=128,
    return_tensors='pt'
)
val_encodings = tokenizer(
    val_df['text'].tolist(),
    truncation=True,
    padding='max_length',
    max_length=128,
    return_tensors='pt'
)

# Tokenize test set
test_encodings = tokenizer(
    test_df['text'].tolist(),
    truncation=True,
    padding='max_length',
    max_length=128,
    return_tensors='pt'
)
# Create datasets (using your EmotionDataset, not ChatDataset from training)
class EmotionDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = torch.tensor(labels, dtype=torch.float)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return {
            'input_ids': self.encodings['input_ids'][idx],
            'attention_mask': self.encodings['attention_mask'][idx],
            'labels': self.labels[idx]
        }


# Tokenize validation set


# Create datasets using pre-tokenized encodings
train_dataset = EmotionDataset(train_encodings, train_labels)
val_dataset   = EmotionDataset(val_encodings, val_labels)
test_dataset  = EmotionDataset(test_encodings, test_labels)

# ============================================================
# FIX FOR IMBALANCED DATA
# ============================================================
print("\n⚖️ Calculating class weights...")
pos_weights = []
for col in emotion_cols:
    pos_count = train_df[col].sum()
    neg_count = len(train_df) - pos_count
    pos_weights.append(neg_count / max(pos_count, 1))

pos_weights = torch.tensor(pos_weights, dtype=torch.float).to(device)

# Create weighted sampler
sample_weights = np.zeros(len(train_dataset))
for i in range(len(train_dataset)):
    pos_count = train_labels[i].sum()
    sample_weights[i] = 1.0 if pos_count == 0 else 1.0 / pos_count

sampler = WeightedRandomSampler(sample_weights, len(sample_weights), replacement=True)

# Create dataloaders
batch_size = 32
train_loader = DataLoader(train_dataset, batch_size=batch_size, sampler=sampler, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=0)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=0)

print(f"Train batches: {len(train_loader)}")
print(f"Val batches: {len(val_loader)}")
print(f"Test batches: {len(test_loader)}")

# Initialize model
print("\nInitializing DistilBERT model...")
model = DistilBertForSequenceClassification.from_pretrained(
    'distilbert-base-uncased',
    num_labels=len(emotion_cols),
    problem_type="multi_label_classification"
).to(device)

# Training setup
epochs = 8
learning_rate = 2e-5
optimizer = AdamW(model.parameters(), lr=learning_rate)
total_steps = len(train_loader) * epochs
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=int(0.1 * total_steps),
    num_training_steps=total_steps
)

# Loss function with class weights
criterion = torch.nn.BCEWithLogitsLoss(pos_weight=pos_weights)

# ============================================================
# TRAINING LOOP (Now properly using the training utilities)
# ============================================================
print("\n" + "="*60)
print("STARTING MULTI-LABEL EMOTION CLASSIFICATION TRAINING")
print("="*60)

best_f1_micro = 0
history = {
    'train_loss': [], 'val_loss': [],
    'train_f1_micro': [], 'val_f1_micro': [],
    'train_f1_macro': [], 'val_f1_macro': []
}

for epoch in range(epochs):
    print(f"\n{'='*50}")
    print(f"Epoch {epoch + 1}/{epochs}")
    print('='*50)

    # Train - using train_epoch from core.training but adapted for multi-label
    model.train()
    total_loss = 0
    all_preds = []
    all_labels = []

    for batch in tqdm(train_loader, desc="Training"):
        optimizer.zero_grad()

        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        outputs = model(input_ids, attention_mask=attention_mask)
        logits = outputs.logits

        loss = criterion(logits, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()

        total_loss += loss.item()

        # Store predictions
        preds = torch.sigmoid(logits) > 0.5
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

    train_loss = total_loss / len(train_loader)
    train_preds = np.array(all_preds)
    train_true = np.array(all_labels)
    train_f1_micro = f1_score(train_true, train_preds, average='micro', zero_division=0)
    train_f1_macro = f1_score(train_true, train_preds, average='macro', zero_division=0)

    # Validate
    model.eval()
    val_preds = []
    val_labels_list = []
    val_loss = 0

    with torch.no_grad():
        for batch in tqdm(val_loader, desc="Evaluating"):
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)

            outputs = model(input_ids, attention_mask=attention_mask)
            logits = outputs.logits

            loss = criterion(logits, labels)
            val_loss += loss.item()

            preds = torch.sigmoid(logits) > 0.5
            val_preds.extend(preds.cpu().numpy())
            val_labels_list.extend(labels.cpu().numpy())

    val_loss = val_loss / len(val_loader)
    val_preds = np.array(val_preds)
    val_true = np.array(val_labels_list)
    val_f1_micro = f1_score(val_true, val_preds, average='micro', zero_division=0)
    val_f1_macro = f1_score(val_true, val_preds, average='macro', zero_division=0)

    # Update history
    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['train_f1_micro'].append(train_f1_micro)
    history['val_f1_micro'].append(val_f1_micro)
    history['train_f1_macro'].append(train_f1_macro)
    history['val_f1_macro'].append(val_f1_macro)

    print(f"\nTrain Loss: {train_loss:.4f}, F1-micro: {train_f1_micro:.4f}")
    print(f"Val Loss: {val_loss:.4f}, F1-micro: {val_f1_micro:.4f}")

    # Save best model
    if val_f1_micro > best_f1_micro:
        best_f1_micro = val_f1_micro
        save_model(model, tokenizer, f'{MODEL_PATH}/best_model')
        print(f"✓ Saved best model (F1-micro={best_f1_micro:.4f})")

# Final evaluation on test set
print("\n" + "="*60)
print("TEST SET EVALUATION")
print("="*60)

model.eval()
test_preds = []
test_labels_list = []
test_loss = 0

with torch.no_grad():
    for batch in tqdm(test_loader, desc="Testing"):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        outputs = model(input_ids, attention_mask=attention_mask)
        logits = outputs.logits

        loss = criterion(logits, labels)
        test_loss += loss.item()

        preds = torch.sigmoid(logits) > 0.5
        test_preds.extend(preds.cpu().numpy())
        test_labels_list.extend(labels.cpu().numpy())

test_preds = np.array(test_preds)
test_true = np.array(test_labels_list)
test_f1_micro = f1_score(test_true, test_preds, average='micro', zero_division=0)
test_f1_macro = f1_score(test_true, test_preds, average='macro', zero_division=0)
per_class_f1 = f1_score(test_true, test_preds, average=None, zero_division=0)

print(f"\nTest Loss: {test_loss/len(test_loader):.4f}")
print(f"Test F1-micro: {test_f1_micro:.4f}")
print(f"Test F1-macro: {test_f1_macro:.4f}")

# Per-class performance
print("\n📊 Per-class F1 Scores (top 10):")
class_f1 = [(emotion_cols[i], per_class_f1[i]) for i in range(len(emotion_cols))]
class_f1.sort(key=lambda x: x[1], reverse=True)

for emotion, f1 in class_f1[:10]:
    print(f"  {emotion}: {f1:.4f}")

# Save results
results = {
    'test_f1_micro': float(test_f1_micro),
    'test_f1_macro': float(test_f1_macro),
    'per_class_f1': [float(f1) for f1 in per_class_f1],
    'best_val_f1_micro': float(best_f1_micro)
}

with open(f'{MODEL_PATH}/test_results.json', 'w') as f:
    json.dump(results, f, indent=2)

print(f"\n✅ All results saved to {MODEL_PATH}")

# Inference examples
print("\n" + "="*60)
print("TESTING INFERENCE")
print("="*60)

def predict_emotions(text, model, tokenizer, emotion_cols, device, threshold=0.5):
    model.eval()
    encoding = tokenizer(text, truncation=True, padding='max_length',
                        max_length=128, return_tensors='pt').to(device)

    with torch.no_grad():
        outputs = model(**encoding)
        probs = torch.sigmoid(outputs.logits)[0].cpu().numpy()

    results = []
    for i, (emotion, prob) in enumerate(zip(emotion_cols, probs)):
        if prob > threshold:
            results.append({'emotion': emotion, 'probability': float(prob)})

    return sorted(results, key=lambda x: x['probability'], reverse=True)




[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Using device: cuda

Train: (109973, 29), Val: (20122, 29), Test: (20164, 29)
Number of emotion classes: 28

Train: (109973, 29), Val: (20122, 29), Test: (20164, 29)
Number of emotion classes: 28

Using 28 emotion classes

⚖️ Calculating class weights...
Train batches: 3437
Val batches: 629
Test batches: 631

Initializing DistilBERT model...


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



STARTING MULTI-LABEL EMOTION CLASSIFICATION TRAINING

Epoch 1/8


...


Train Loss: 0.5530, F1-micro: 0.3404
Val Loss: 0.9620, F1-micro: 0.3138


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved to C:/Child Guardian GP2/models/emotions_distilbert/best_model
✓ Saved best model (F1-micro=0.3138)

TEST SET EVALUATION


Testing: 100%|██████████| 631/631 [00:11<00:00, 56.99it/s]



Test Loss: 0.9523
Test F1-micro: 0.3120
Test F1-macro: 0.2791

📊 Per-class F1 Scores (top 10):
  gratitude: 0.5671
  love: 0.5278
  neutral: 0.5066
  admiration: 0.4754
  amusement: 0.4750
  curiosity: 0.3798
  sadness: 0.2887
  remorse: 0.2742
  anger: 0.2707
  joy: 0.2700

✅ All results saved to C:/Child Guardian GP2/models/emotions_distilbert

TESTING INFERENCE


In [ ]:
test_texts = [
    # 🔴 Quiet distress (not obvious)
    "I’ve been really tired lately, just don’t feel like myself.",
    "Everything feels kind of heavy these days.",
    "I don’t really see the point in trying anymore.",
    "I’ve been staying in a lot, just don’t have the energy.",
    "I keep messing things up no matter what I do.",
    "Some days it’s just hard to get out of bed.",
    "I’ve been thinking a lot and none of it feels good.",
    "I don’t talk to people much anymore.",
    "I just feel off, like something’s wrong but I can’t explain it.",
    "It’s getting harder to act like everything’s fine.",

    # 🟡 everyday emotional struggles
    "I’ve been stressed about everything recently.",
    "I keep overthinking small things for no reason.",
    "I feel like I’m falling behind compared to everyone else.",
    "Today was just one of those days.",
    "I’m not in the mood for anything lately.",
    "I get annoyed so easily these days.",
    "I feel kind of disconnected from everything.",
    "I’ve been worrying a lot about the future.",
    "I don’t really feel excited about things anymore.",
    "I wish I could just reset everything.",

    # 🟢 normal / positive (but still natural)
    "Today was actually pretty nice.",
    "I enjoyed hanging out earlier.",
    "I feel a bit better than yesterday.",
    "Things are slowly getting better I think.",
    "I had a good laugh today.",
    "I’m kind of looking forward to tomorrow.",
    "It was a calm day, nothing crazy.",
    "I finally got something done that I was putting off.",
    "I feel okay today, not great but okay.",
    "Spent some time outside, felt nice.",

    # 😐 neutral / casual
    "I need to finish some work later.",
    "I’ve got a lot to do this week.",
    "Just woke up not too long ago.",
    "I’ve been on my phone way too much today.",
    "Not sure what I’m doing tonight.",
    "Just another normal day honestly.",
    "I might go out later, haven’t decided yet.",
    "I’ve been kind of busy lately.",
    "I need to fix my sleep schedule.",
    "I’ve just been chilling most of the day.",

    # 🧠 ambiguous (the important ones)
    "I’m fine, just tired.",
    "It is what it is I guess.",
    "I don’t really care anymore.",
    "Same as always.",
    "I’ll deal with it somehow.",
    "I’m used to it by now.",
    "Nothing new really.",
    "I guess things could be worse.",
    "I don’t feel much about it.",
    "It’s whatever honestly.",

    # 😡 subtle strong emotions
    "That really annoyed me more than it should have.",
    "I didn’t like how that went at all.",
    "I feel kind of stupid for that.",
    "That was honestly frustrating.",
    "I’m a bit upset about earlier.",
    "That made me uncomfortable.",
    "I wish I handled that differently.",
    "I’m still thinking about what happened.",
    "That didn’t sit right with me.",
    "I don’t think I reacted well to that."
]

for text in test_texts:
    preds = predict_emotions(text, model, tokenizer, emotion_cols, device)
    print(f"\nText: {text}")
    for pred in preds[:3]:
        print(f"  - {pred['emotion']}: {pred['probability']:.3f}")


Text: I’ve been really tired lately, just don’t feel like myself.
  - nervousness: 0.968
  - sadness: 0.927
  - disappointment: 0.820

Text: Everything feels kind of heavy these days.
  - nervousness: 0.915
  - sadness: 0.847
  - disappointment: 0.820

Text: I don’t really see the point in trying anymore.
  - disapproval: 0.880
  - confusion: 0.866
  - neutral: 0.611

Text: I’ve been staying in a lot, just don’t have the energy.
  - disappointment: 0.826
  - disapproval: 0.707
  - realization: 0.651

Text: I keep messing things up no matter what I do.
  - annoyance: 0.889
  - approval: 0.664
  - neutral: 0.578

Text: Some days it’s just hard to get out of bed.
  - sadness: 0.940
  - nervousness: 0.872
  - disappointment: 0.838

Text: I’ve been thinking a lot and none of it feels good.
  - disappointment: 0.901
  - sadness: 0.789
  - disapproval: 0.742

Text: I don’t talk to people much anymore.
  - disapproval: 0.843
  - annoyance: 0.812
  - anger: 0.751

Text: I just feel off, like s

In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score

print("\n📋 Creating performance table...")

precision = precision_score(test_true, test_preds, average=None, zero_division=0)
recall    = recall_score(test_true, test_preds, average=None, zero_division=0)
f1        = f1_score(test_true, test_preds, average=None, zero_division=0)

results_df = pd.DataFrame({
    'Emotion': emotion_cols,
    'Precision': precision,
    'Recall': recall,
    'F1 Score': f1
})

# Sort by F1
results_df = results_df.sort_values(by='F1 Score', ascending=False)

print("\n📊 Top 10 emotions:")
print(results_df)

# Save full table
results_df.to_csv(f'{MODEL_PATH}/detailed_metrics.csv', index=False)


📋 Creating performance table...

📊 Top 10 emotions:
           Emotion  Precision    Recall  F1 Score
15       gratitude   0.416743  0.887365  0.567136
18            love   0.382105  0.853015  0.527789
27         neutral   0.414518  0.651306  0.506609
0       admiration   0.338945  0.795870  0.475419
1        amusement   0.330916  0.841014  0.474951
7        curiosity   0.244695  0.848384  0.379835
25         sadness   0.177320  0.775940  0.288671
24         remorse   0.170958  0.691667  0.274154
2            anger   0.166667  0.721019  0.270749
17             joy   0.164365  0.756036  0.270025
4         approval   0.172075  0.596653  0.267114
10     disapproval   0.163311  0.726052  0.266645
20        optimism   0.163842  0.680751  0.264117
14            fear   0.162595  0.699275  0.263841
26        surprise   0.162114  0.702913  0.263464
6        confusion   0.153846  0.772664  0.256600
3        annoyance   0.152420  0.696924  0.250135
5           caring   0.145486  0.711375  0.2415